# 🤘 Cache Me If You Can — Act 0: hello, chip

Behind this browser tab is a Tenstorrent **n150** card, reserved for you for the morning: a chip with a grid of compute cores and 12 GB of memory.

This notebook does three things:

1. Checks that your card works (30 seconds).
2. Starts the **warm-up** in the background (explained below). It finishes during the first talk.
3. Introduces three words you will hear all morning.
4. Shows where to look up how a TT-NN function is called.

The model weights are already on this machine. Nothing is downloaded during the workshop. There is nothing to fill in here: run the cells from top to bottom.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()                  # the folder that holds common/ and notebooks/
while ROOT != ROOT.parent and not (ROOT / "common" / "speedrun.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from common import config, wait_for_warmup
wait_for_warmup()
import os, socket, subprocess
import torch, ttnn
print("TT-NN loaded from", Path(ttnn.__file__).parent)

## 1. Roll call

Open the card, run one small operation on it, and read the result back. (The first line of the cell runs it in a separate, short-lived process; Act 0 explains why below.) If the number is right, your card and the software are talking to each other.

The roll call also prints how many compute cores programs can use on your card. On a Wormhole card (Tenstorrent's 2nd generation, such as the n150) expect 64 or 72; on a Blackhole card (3rd generation) 110 or 120. The exact number depends on the card model.

In [ ]:
%%script {sys.executable}
# This cell runs in its own short-lived Python process: a program that has used the card keeps it
# until it stops, and the warm-up started below needs the card next.
import os, socket, torch, ttnn
device = ttnn.open_device(device_id=0)
a = ttnn.from_torch(torch.full((32, 32), 20.0), layout=ttnn.TILE_LAYOUT, device=device)
b = ttnn.from_torch(torch.full((32, 32), 6.0), layout=ttnn.TILE_LAYOUT, device=device)
answer = ttnn.to_torch(ttnn.multiply(a, b))[0, 0].item()
grid = device.compute_with_storage_grid_size()
print(f"{os.environ.get('MLINPL_SEAT', socket.gethostname())} online: {device.arch()}, "
      f"{grid.x} x {grid.y} = {grid.x * grid.y} compute cores available, 20 x 6 = {answer:.0f}")
assert answer == 120.0, "the card disagrees about 20 x 6 - please call a presenter"
ttnn.close_device(device)

## 2. The warm-up

Each core of the chip runs small programs called **kernels**. Usually a kernel performs one basic operation, such as an addition, a matrix multiplication, a softmax or a normalization. More complex kernels combine several, for example a matrix multiplication and its activation function in one program.

TT-NN **compiles** a kernel the first time an operation runs with a given tensor shape, and stores the compiled kernel on disk. Compiling everything a model needs takes minutes, but every later run reuses the stored kernels and starts in seconds.

Running the models once ahead of time, only so that their kernels get compiled, is what we call the **warm-up**. It comes in two parts, each running in the background:

- **Part 1**, started by the cell below, prepares the language model of Act I. It finishes during the first talk.
- **Part 2** starts at the end of Act I and prepares everything else (GPT-2 for Acts II and IV, the settings of Act III) during the break.

How long it takes on an n150:

| | First time on a machine | Once compiled |
|---|---|---|
| Part 1 | about 2 minutes | about 25 seconds |
| Part 2 | about 9 minutes | about 1.5 minutes |

Each part writes its progress to a log in the `notebooks` folder: part 1 to `warmup.log`, part 2 to `warmup2.log`. Every finished step adds a line such as `[warm-up 09:12:40] language model {'max_batch_size': 1} ready (127 s)`, and the last line is `WARMUP DONE`, or `WARMUP FAILED` followed by the error. Lines marked `warning` come from TT-NN itself and are harmless. If a log gets no new step line for much longer than the times above, something is stuck.

**One rule for the whole morning:** the card can serve only one process at a time, and it stays with that process until the process ends. Every notebook runs as its own process (its kernel). That is why the roll call ran in a separate process, and why the last cell of every act stops that notebook's kernel.

In [ ]:
from common import start_warmup
start_warmup(1)

It keeps running even if you close this notebook. Check on it whenever you like with the next cell, or in a terminal with `tail -f notebooks/warmup.log`. Part 1 should end with `WARMUP DONE` within a few minutes. If it ends with an error, call a presenter rather than restarting it.

In [ ]:
for log in config.WARMUP_LOGS.values():
    if log.exists():
        print(f"--- {log.name}", *log.read_text().splitlines()[-4:], sep="\n")

## 3. Three words for the morning

- A **token** is a piece of text, usually a word or part of a word. Models read and write text as tokens, and each token is a number in the model's vocabulary. The cell below shows how a sentence is split.
- **Time to first token (TTFT):** the time from sending a prompt until the first token of the answer arrives.
- **Tokens per second per user:** once the answer has started, how many tokens per second arrive for one user.

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(config.local_weights(config.LLM_MODEL_ID))   # the Act I model's tokenizer
sentence = "Cache Me If You Can: a Tenstorrent workshop at ML in PL"
ids = tokenizer.encode(sentence, add_special_tokens=False)
print(len(ids), "tokens:", ids)
print([tokenizer.decode([i]) for i in ids])

## 4. Looking things up

From Act II on you write TT-NN code yourself. Three ways to find out how a function is called:

- **TT-NN follows PyTorch.** Most functions have the same names and arguments: `ttnn.matmul(a, b)`, `ttnn.add(a, b)`, `ttnn.softmax(x, dim=-1)`, `ttnn.permute(x, dims)`, `ttnn.concat([a, b], dim=...)`. If you know the PyTorch version, try the same in TT-NN.
- **`help()` in any cell**, for example `help(ttnn.permute)`, shows the documentation of the TT-NN version installed here.
- **The online reference:** [TT-NN API](https://docs.tenstorrent.com/tt-metal/latest/ttnn/ttnn/api.html). It describes the newest release; if it disagrees with `help()`, trust `help()`.